In [5]:
import sys
import io
import requests
from pathlib import Path
sys.path.append("/home/millieginty/OneDrive/git-repos/LCBP-interannual-EMMAs/")  

import importlib
import pandas as pd
import EMMA.event_emma as em
importlib.reload(em)

# Import uncertainty propagation module
import EMMA.uncertainty_propagation as up
importlib.reload(up)

<module 'EMMA.uncertainty_propagation' from '/home/millieginty/OneDrive/git-repos/LCBP-interannual-EMMAs/EMMA/uncertainty_propagation.py'>

In [13]:
# Load the full RI25 dataset
df = pd.read_csv("/home/millieginty/OneDrive/git-repos/EMMA/data/newrnet-chemistry/RI23/RI23-IC-ICP-isotope-toc-joined-Wade.csv")

####################
# Wade RI23 events #
####################

wade_tracers = ['Ca_mg_L', 'Si_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']

# 1. Run your existing EMMA function
(
    wade_febros_fractions_df,
    wade_febros_scaler,
    wade_febros_pca,
    wade_febros_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Wade",
    start_date="2023-02-14 00:00:00",
    end_date="2023-02-20 00:00:00",
    endmember_ids=["RI23-5006", "RI23-5018", "RI23-5000", "RI23-5005"],
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin(["RI23-5006", "RI23-5018", "RI23-5000", "RI23-5005"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Wade")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-02-14 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-02-20 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties (Optional: edit these to match lab detection limits)
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# 5. Calculate Genereux Uncertainties
uncertainty_df = up.propagate_genereux_uncertainty(
    stream_df=stream_event_df,
    em_grouped=wade_febros_endmembers_df,
    em_raw=em_raw_subset,
    tracers=wade_tracers,
    analytical_sd=analytical_sd,
)

# 6. Merge fractions and their calculated uncertainties for a complete table
results_with_error = pd.merge(
    wade_febros_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
)

# Identify which uncertainty columns were actually generated
uncertainty_cols = [
    col for col in results_with_error.columns if "Uncertainty_1sig" in col
]
# Extract the base fraction names (e.g., "Groundwater", "Snowmelt lysimeter")
fraction_cols = [col.replace("_Uncertainty_1sig", "") for col in uncertainty_cols]

# Combine them in alternating order: [Fraction_1, Uncertainty_1, Fraction_2, Uncertainty_2...]
display_cols = ["Sample ID"]
for frac, unc in zip(fraction_cols, uncertainty_cols):
    display_cols.extend([frac, unc])

# Print the head of the dynamically built column list
print(results_with_error[display_cols].head())

   Sample ID  Groundwater  Groundwater_Uncertainty_1sig  Snowmelt lysimeter  \
0  RI23-1009     0.577292                      0.119733        3.651759e-14   
1  RI23-1010     0.531795                      0.120108       -5.082566e-14   
2  RI23-1011     0.495768                      0.120825       -6.959312e-15   
3  RI23-1025     0.524279                      0.122459       -1.318793e-13   
4  RI23-1012     0.326576                      0.126350        2.709334e-14   

   Snowmelt lysimeter_Uncertainty_1sig  Soil water lysimeter dry  \
0                         2.818795e-07                  0.422708   
1                         2.683066e-10                  0.468205   
2                         1.410120e-09                  0.504232   
3                         2.348696e-07                  0.475721   
4                         1.757562e-09                  0.673424   

   Soil water lysimeter dry_Uncertainty_1sig  
0                                   0.119733  
1                     

In [10]:
print(results_with_error.columns.tolist())

['Sample ID', 'Datetime', 'Site', 'Groundwater', 'Snowmelt lysimeter', 'Soil water lysimeter dry', 'Sum_Fractions', 'Groundwater_Uncertainty_1sig', 'Snowmelt lysimeter_Uncertainty_1sig', 'Soil water lysimeter dry_Uncertainty_1sig']
